In [0]:
%sql
CREATE DATABASE IF NOT EXISTS bronze_db;
SHOW DATABASES;

In [0]:
%sql
USE bronze_db;

In [0]:
%sql
SHOW VOLUMES

In [0]:
# Script d'ingestion des données
from pyspark.sql.functions import current_timestamp, col, substring_index
from pyspark.sql.types import StructType, StructField, IntegerType, LongType, DoubleType, StringType, TimestampType
import datetime

# 1. Définir le schema des données
schema = StructType([
  StructField("VendorID", IntegerType(), True),
  StructField("tpep_pickup_datetime", TimestampType(), True),
  StructField("tpep_dropoff_datetime", TimestampType(), True),
  StructField("passenger_count", LongType(), True),
  StructField("trip_distance", DoubleType(), True),
  StructField("RatecodeID", LongType(), True),
  StructField("store_and_fwd_flag", StringType(), True),
  StructField("PULocationID", IntegerType(), True),
  StructField("DOLocationID", IntegerType(), True),
  StructField("payment_type", LongType(), True),
  StructField("fare_amount", DoubleType(), True),
  StructField("extra", DoubleType(), True),
  StructField("mta_tax", DoubleType(), True),
  StructField("tip_amount", DoubleType(), True),
  StructField("tolls_amount", DoubleType(), True),
  StructField("improvement_surcharge", DoubleType(), True),
  StructField("total_amount", DoubleType(), True),
  StructField("congestion_surcharge", DoubleType(), True),
  StructField("Airport_fee", DoubleType(), True)
])

# 2. Dossier source
sourceFolder = "/Volumes/workspace/bronze_db/data/"
filePath = sourceFolder + "*.parquet"

# 3. Lister les fichiers parquet
files = [file for file in dbutils.fs.ls(sourceFolder) if file.name.endswith(".parquet")]
if files:
    print(f"{len(files)} fichiers trouvés, ingestion en cours...")
    # 4. Lire les fichiers avec schéma forcé
    df = spark.read.schema(schema).parquet(filePath)
    display(df)

    # 5. Ajouter une colonne ingestion timestamp + une colonne nom de fichier
    df = df.withColumn("ingestion_timestamp", current_timestamp()) \
           .withColumn("file_name", substring_index(col("_metadata.file_path"), "/", -1))

    # 6. Calcul des nombres de lignes
    # 6.1 Nombre de lignes dans la dataframe pyspark ingéré
    new_rows_count = df.count()

    # 6.2 Vérifier si la table existe dans le catalogue spark
    if "raw_trips"  in [table.name for table in spark.catalog.listTables("bronze_db")]:
        existing_rows_count = spark.table("bronze_db.raw_trips").count()
    else:
        existing_rows_count = 0

    # 6.3 Affichage des informations
    print(f"Nombre de lignes ingéré : {new_rows_count}")
    print(f"Nombre de lignes existante dans la table : {existing_rows_count}")

    # 7. Sauvegarde des données dans une table Delta
    df.write.format("delta").mode("append").option("mergeSchema", True).saveAsTable("raw_trips")

    # 8. Contrôle de cohérence au niveau des nombres de lignes
    final_rows_count = spark.table("bronze_db.raw_trips").count()
    expected_rows = existing_rows_count + new_rows_count

    if final_rows_count != expected_rows:
        raise Exception(f"Contrôle de cohérence échoué : le nombre de lignes dans la table finale {final_rows_count} ne correspond pas au nombre de lignes attendues {expected_rows}")
    else: 
        print(f"Contrôle de cohérence réussi : le nombre de lignes dans la table finale {final_rows_count} correspond au nombre de lignes attendu {expected_rows}")
    
    # 9. Archiver les fichiers parquet traités
    archiveFolder = sourceFolder + "archive/run_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S") + "/"
    for file in files:
        dbutils.fs.mv(file.path, archiveFolder + file.name)
    print(f"{len(files)} fichiers archivés dans {archiveFolder}")

    # 10. Afficher le contenu du dossier archive
    display(dbutils.fs.ls(archiveFolder))

else:
    print("Aucun fichier parquet trouvé dans le dossier source")


In [0]:
%sql
SELECT * FROM raw_trips LIMIT 10

In [0]:
%sql
SELECT DISTINCT file_name FROM raw_trips 
ORDER BY file_name

In [0]:
%sql
DESCRIBE EXTENDED raw_trips

In [0]:
%fs ls /Volumes/workspace/bronze_db/data/

In [0]:
zone_file_path = "/Volumes/workspace/bronze_db/data/taxi_zone_lookup.csv"
df_zones = spark.read.option("header", True).option("inferSchema", True).csv(zone_file_path)
display(df_zones)

In [0]:
df_zones.write.format("delta").mode("ignore").saveAsTable("taxi_zones")

In [0]:
%sql
SELECT * FROM taxi_zones LIMIT 10